## Step1 - Load the PDF
We open the PDF and extract all text from every page into one big string.

In [3]:
import fitz                          # PyMuPDF — reads PDFs
import chromadb                       # vector database
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os

load_dotenv()

KeyError: '__reduce_cython__'

In [5]:
def load_pdf(path):
    doc = fitz.open(path)         # open the PDF file
    full_text = ""
    for page in doc:
        full_text += page.get_text() # extract text from each page
    return full_text

PDF_PATH = "../docs/osnotes.pdf"
raw_text = load_pdf(PDF_PATH)

print(f"Total characters: {len(raw_text)}")
print("\nFirst 500 characters:")
print(raw_text[:500])

Total characters: 106399

First 500 characters:
24TU05MJC2 
Operating System 
L T  P  C 
Version 1.0 
 
2 1 0 3 
Pre-requisites/Exposure 
Data Structures Algorithm  
Co-requisites 
 
 
COURSE OBJECTIVES 
• To understand the fundamental concepts, architecture, and functions of modern 
operating systems. 
• To explore process management, CPU scheduling, synchronization, and deadlock 
handling mechanisms. 
• To study the principles of memory, file, and I/O management in operating systems. 
• To apply theoretical knowledge through simulation, cas


## Step 2 - Chunk the text
LLMs have a limited context window — they can't read your entire PDF at once. We split it into small overlapping pieces called chunks.

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # each chunk is ~600 characters
    chunk_overlap=100,   # 100 chars shared between chunks
    separators=["\n\n", "\n", ".", " "]  # split at paragraph > line > sentence > word
)

chunks = splitter.split_text(raw_text)

print(f"Total chunks created: {len(chunks)}")
print(f"\n--- Chunk 0 ---\n{chunks[0]}")
print(f"\n--- Chunk 1 ---\n{chunks[1]}")
print(f"\nNotice the overlap — chunk 1 starts slightly before chunk 0 ends")

Total chunks created: 217

--- Chunk 0 ---
24TU05MJC2 
Operating System 
L T  P  C 
Version 1.0 
 
2 1 0 3 
Pre-requisites/Exposure 
Data Structures Algorithm  
Co-requisites 
 
 
COURSE OBJECTIVES 
• To understand the fundamental concepts, architecture, and functions of modern 
operating systems. 
• To explore process management, CPU scheduling, synchronization, and deadlock 
handling mechanisms. 
• To study the principles of memory, file, and I/O management in operating systems. 
• To apply theoretical knowledge through simulation, case studies, and practical 
problem-solving related to OS design. 
COURSE OUTCOMES (COS)

--- Chunk 1 ---
problem-solving related to OS design. 
COURSE OUTCOMES (COS) 
After successful completion of the course, the students will be able to: 
• Explain the structure, functions, and components of operating systems.  
• Apply process scheduling and synchronization techniques to improve system 
performance. 
• Analyze and evaluate memory, file, and I/O manage

## Step 3 - Create embeddings
converting each chunk into a vector — a list of 384 numbers that represents its meaning. Chunks with similar meaning will have similar vectors.

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")
# downloads ~80MB model on first run, cached after that

embeddings = model.encode(
    chunks,
    show_progress_bar=True,  # shows a progress bar
    batch_size=32            # processes 32 chunks at a time
)

print(f"Embeddings shape: {embeddings.shape}")
# expect: (num_chunks, 384) — 384 numbers per chunk

print(f"\nSample — first 10 numbers of chunk 0's embedding:")
print(embeddings[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Z\Project\Docknows\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nirmal Choyal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (217, 384)

Sample — first 10 numbers of chunk 0's embedding:
[-0.03073432  0.01290357 -0.02018091 -0.02867743  0.03137984 -0.11081437
  0.01091352  0.07822128 -0.04241128  0.0323217 ]


## Step 4 - Store in ChromaDB
saving all chunks and their embeddings into a local vector database can search them later without re-embedding every time.

In [8]:
client = chromadb.PersistentClient(path="../chroma_db")
# PersistentClient saves to disk — data survives after you close the notebook

try:
    client.delete_collection("my_docs")  # clear old data if re-running
except:
    pass

collection = client.create_collection("my_docs")

collection.add(
    documents=chunks,                    # the actual text
    embeddings=embeddings.tolist(),    # the vectors
    ids=[f"chunk_{i}" for i in range(len(chunks))]  # unique ID per chunk
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 217 chunks in ChromaDB


## Step 5 - Test retrieval
Asking a question and verifying the right chunks come back. This is for sanity check.

In [9]:
query = "What are the generation of OS?"
# change this to something specific from YOUR pdf

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3   # retrieve top 3 most relevant chunks
)

for i, chunk in enumerate(results["documents"][0]):
    print(f"\n{'='*50}")
    print(f"Chunk {i+1} (ID: {results['ids'][0][i]})")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    # lower distance = more similar to query
    print(chunk[:400])


Chunk 1 (ID: chunk_28)
Distance: 0.6666
• In the late 1960s, the first version of the Unix OS was developed 
• The first OS built by Microsoft was DOS. It was built in 1981 by purchasing 
the 86-DOS software from a Seattle company 
• The present-day popular OS Windows first came to existence in 1985 when 
a GUI was created and paired with MS-DOS. 
Types of Operating Systems 
An Operating System performs all the basic tasks like managing

Chunk 2 (ID: chunk_2)
Distance: 0.7811
part of a team-based project and communicate technical findings clearly. 
 
UNIT I – Introduction to Operating Systems                                                                6 Hours                        
Concept and Functions of Operating Systems, Evolution and Generations of Operating 
Systems, Types of Operating Systems: Batch, Multiprogramming, Time Sharing, Real-Time, 
Operating Syst

Chunk 3 (ID: chunk_13)
Distance: 0.8062
The operating system is a set of special programs that run on a computer sys